In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import os
import zipfile

In [ ]:
if not os.path.exists("images"):
    with zipfile.ZipFile("images.zip", "r") as z:
        z.extractall(".")

<div class="alert alert-warning">

# PS 6 - k-Means Clustering

In this problem set, we will implement **k-means clustering**, an unsupervised learning algorithm that groups data points into clusters based on their similarity.

## The Problem

An example observation from the categories cat, dog, and mop are presented below:

<img src="images/test.png"/>

We assume each observation can be represented as a pair of ($x,y$) coordinates, i.e., each object is represented in two-dimensional space. Suppose we have observed some instances of each type of object, but have lost the information as to which instance belongs to which type!

To try and recover this information we will use an unsupervised learning algorithm called **k-means clustering**. The $k$ here refers to how many types of clusters we think exist in the data, and the goal of the algorithm is to assign labels to the data points using their distance to the centers (or means) of the clusters.

## The Algorithm

For this problem, we assume $k=3$. After randomly initializing cluster centers, the algorithm alternates between two steps:

1. **Assignment step**: Update the label assignments of the data points based on the nearest cluster centers
2. **Update step**: Update the positions of the cluster centers to reflect the updated assignments of data points

</div>

Before you begin, load the data we will be using:

<div class="alert alert-info">

**Note:** We use non-random initializations for the cluster centers to make the test cases work correctly. Normally cluster centers would be randomly initialized, and we will implement this in later questions.

</div>

In [ ]:
if not os.path.exists('data/X.npz'):
    with zipfile.ZipFile('data.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
        
data = np.load('data/X.npz')
X = data['X']
centers = data['centers'] 

print('X: \n' + str(X))
print('\ncenters: \n' + str(centers))

---

# 1. Computing Distance

First, we will need a function that gives us the distance between two points. We can use **Euclidean distance** to compute the distance between two points ($x_1,y_1$) and ($x_2,y_2$). Recall that Euclidean distance in $\mathbb{R}^2$ is calculated as:

$$\text{distance}((x_1,y_1),(x_2,y_2)) = \sqrt{(x_1 - x_2)^{2} + (y_1 - y_2)^{2}}$$

<div class="alert alert-success">

## Exercise 1

Complete the `distance` function below to calculate the Euclidean distance between two points in $\mathbb{R}^2$.

</div>

In [ ]:
def distance(a, b):
    """
    Returns the Euclidean distance between two points, 
    a and b, in R^2.
    
    Parameters
    ----------
    a, b : numpy arrays of shape (2,)
        The (x,y) coordinates for two points, a and b, 
        in R^2. E.g., a[0] is the x coordinate, 
        and a[1] is the y coordinate.
            
    Returns
    -------
    distance : float
        The Euclidean distance between a and b
    """
    # YOUR CODE HERE

In [ ]:
# add your own test cases here!


In [ ]:
"""Check distance computes the correct values"""
from numpy.testing import assert_allclose

assert_allclose(distance(np.array([0.0, 0.0]), np.array([0.0, 1.0])), 1.0)
assert_allclose(distance(np.array([3.0, 3.0]), np.array([4.3, 5.0])), 2.3853720883753127)
assert_allclose(distance(np.array([130.0, -25.0]), np.array([0.4, 15.0])), 135.63244449614552)

print("Success!")

---

# 2. Assigning Points to Clusters

<div class="alert alert-success">

## Exercise 2

Now, we will write a function to update the cluster that each point is assigned to by computing the distance to the center of each cluster. Complete the `update_assignments` function to do this using your `distance` function.

</div>

In [ ]:
def update_assignments(num_clusters, X, centers):
    """
    Returns the cluster assignment (number) for each data point 
    in X, computed as the closest cluster center.
    
    Parameters
    ----------
    num_clusters : int
        The number of disjoint clusters (i.e., k) in 
        the X
    
    X : numpy array of shape (m, 2)
        An array of m data points in R^2.
    
    centers : numpy array of shape (num_clusters, 2)
        The coordinates for the centers of each cluster
        
    Returns
    -------
    cluster_assignments : numpy array of shape (m,)
        An array containing the cluster label assignments 
        for each data point in X. Each cluster label is an integer
        between 0 and (num_clusters - 1). 
    """
    # YOUR CODE HERE

In [ ]:
# add your own test cases here!


In [ ]:
"""Check update_assignments computes the correct values"""
from numpy.testing import assert_equal, assert_array_equal

# load the data
data = np.load('data/X.npz')
X = data['X']

# validate update_assignments using different values
actual = update_assignments(2, X, np.array([[3, 2], [1, 4]]))
expected = np.array([
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 
    0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0])

# is the output of the correct shape?
assert_equal(actual.shape[0], X.shape[0])

# are the cluster labels correct?
assert_array_equal(expected, actual)

# validate update_assignments using different values
actual = update_assignments(3, X[:int(X.shape[0]/2)], np.array([X[0], X[1], X[2]]))
expected = np.array([0, 1, 2, 2, 0, 2, 1, 2, 2, 2, 0, 0, 0, 0, 0])

# is the output of the correct shape?
assert_equal(actual.shape[0], X.shape[0] / 2)

# are the cluster labels correct?
assert_array_equal(expected, actual)

# check that it uses distance
old_distance = distance
del distance
try:
    update_assignments(2, X, np.array([[3, 2], [1, 4]]))
except NameError:
    pass
else:
    raise AssertionError("update_assignments does not call distance")
finally:
    distance = old_distance
    del old_distance

print("Success!")

---

# 3. Updating Cluster Centers

<div class="alert alert-success">

## Exercise 3

Now, we need to do the next step of the clustering algorithm: recompute the cluster centers based on which points are assigned to that cluster. 

The new centers are simply the **two-dimensional means** of each group of data points. A two-dimensional mean is calculated by finding the mean of the x coordinates and the mean of the y coordinates. 

Complete the `update_parameters` function to do this.

</div>

In [ ]:
def update_parameters(num_clusters, X, cluster_assignment):
    """
    Recalculates cluster centers after running update_assignments.
    
    Parameters
    ----------
    num_clusters : int
        The number of disjoint clusters (i.e., k) in 
        the X
    
    X : numpy array of shape (m, 2)
        An array of m data points in R^2
    
    cluster_assignment : numpy array of shape (m,)
        The array of cluster labels assigned to each data 
        point as returned from update_assignments
    
    Returns
    -------
    updated_centers : numpy array of shape (num_clusters, 2)
        An array containing the new positions for each of 
        the cluster centers
    """
    # YOUR CODE HERE

In [ ]:
# add your own test cases here!


In [ ]:
"""Check update_parameters computes the correct values"""
from numpy.testing import assert_equal
from numpy.testing import assert_allclose

# load the data
data = np.load('data/X.npz')
X = data['X']

# validate update_assignments using different values
cluster_assignment1 = np.array([
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 
    0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0])
actual = update_parameters(2, X, cluster_assignment1)
expected = np.array([[ 3.24286584,  2.71362623], [ 2.80577245,  4.07633606]])
assert_allclose(expected, actual)

cluster_assignment2 = np.array([0, 1, 2, 2, 0, 2, 1, 2, 2, 2, 0, 0, 0, 0, 0])
actual = update_parameters(3, X[:int(X.shape[0]/2)], cluster_assignment2)
expected = np.array([[ 3.4914304 ,  2.79181724], [ 3.03095255,  2.02958778], [ 2.86686881,  1.76070598]])
assert_allclose(expected, actual, rtol=1e-6)
    
print("Success!")

---

# 4. Running k-means

At this stage you are ready to run the k-means clustering algorithm! The `k_means` function below will call your functions from the previous exercises to run the k-means algorithm on the data points in `X`. Note that for this problem we assume that $k = 3$.

Take a look at the function `k_means` we're providing below, and run the next cell to call it.

In [ ]:
def k_means(num_clusters, X, centers, update_assignments, 
            update_parameters, n_iter=4, plotting=True):
    """
    Runs the k-means algorithm for n_iter iterations and plots
    the results.
    
    Parameters
    ----------
    num_clusters : int
        The number of disjoint clusters (i.e., k) to search for
        in X
    
    X : numpy array of shape (m, 2)
        An array of m data points in R^2
        
    centers : numpy array of shape (num_clusters, 2)
        The coordinates for the centers of each cluster
        
    update_assignments : function
        The function you completed in Exercise 2
    
    update_parameters : function
        The function you completed in Exercise 3
    
    n_iter : int (optional)
        The number of iterations to run the k_means update procedure.
        If not specified, defaults to 4.
    
    plotting : boolean (optional)
        Plot the steps if True
        
    Returns
    -------
    cluster_assignments : numpy array of shape (m,) 
        The final label assignments for each of the data points in X
        
    centers : numpy array of shape (num_clusters, 2)
        The final cluster centroids in R^2 after running k-means
    """
    if plotting:
        fig, ax = plt.subplots(2, n_iter, figsize=(4*n_iter, 8), constrained_layout=True)
        ax = ax.flatten()

    for i in range(n_iter):
        # Step 1: Update cluster assignments
        cluster_assignments = \
            update_assignments(num_clusters, X, centers)
           
        if plotting:
            # plot data with colors corresponding to cluster assignments
            colors = ['r', 'b', 'g', 'orange', 'purple', 'cyan']
            for j in range(X.shape[0]):
                c_idx = int(cluster_assignments[j]) % len(colors)
                ax[2*i].plot(X[j,0], X[j,1], colors[c_idx] + 'o', markersize=8)

            # plot the centers as stars with the associated color
            for c_idx in range(len(centers)):
                ax[2*i].plot(centers[c_idx, 0], centers[c_idx, 1], 
                            colors[c_idx % len(colors)] + '*', markersize=20, 
                            markeredgecolor='black', markeredgewidth=1)
            
            ax[2*i].set_title(f'Step 1: Assign Points\nIteration {i+1}', fontsize=12, fontweight='bold')
            ax[2*i].set_xlim([2, 4.5])
            ax[2*i].set_ylim([1, 4.5])
            ax[2*i].set_xticks([]) 
            ax[2*i].set_yticks([])
            ax[2*i].set_aspect('equal')
        
        # Step 2: Update the cluster centers
        centers = \
            update_parameters(num_clusters, X, cluster_assignments)
        
        if plotting:
            # Plot data assignments with the updated center positions
            for j in range(X.shape[0]):
                c_idx = int(cluster_assignments[j]) % len(colors)
                ax[2*i+1].plot(X[j,0], X[j,1], colors[c_idx] + 'o', markersize=8)

            # Plot cluster centers as stars
            for c_idx in range(len(centers)):
                ax[2*i+1].plot(centers[c_idx, 0], centers[c_idx, 1], 
                              colors[c_idx % len(colors)] + '*', markersize=20,
                              markeredgecolor='black', markeredgewidth=1)
                
            ax[2*i+1].set_title(f'Step 2: Update Centers\nIteration {i+1}', fontsize=12, fontweight='bold')
            ax[2*i+1].set_xlim([2, 4.5])
            ax[2*i+1].set_ylim([1, 4.5])
            ax[2*i+1].set_xticks([])
            ax[2*i+1].set_yticks([])
            ax[2*i+1].set_aspect('equal')
        
    if plotting:
        plt.suptitle('K-Means Clustering Algorithm', fontsize=14, fontweight='bold', y=1.05)
        plt.show()
        return cluster_assignments, centers, fig
    else: 
        return cluster_assignments, centers

In [ ]:
# load the data
data = np.load('data/X.npz')
X = data['X']
centers = data['centers'] 

# run k-means
cluster_assignments, updated_centers, fig = k_means(3, X, centers, update_assignments, update_parameters, n_iter=4)

If the functions you completed above are working properly, you should see a figure containing a subplot of the output from steps (1) and (2) for four iterations of the algorithm. This plot should give you a sense of how the algorithm progresses over time. The data points are each assigned to one of three colors corresponding to their current cluster label. The cluster centers are plotted as stars.

<div class="alert alert-success">

## Exercise 4

Describe what happened in the 8 iterations/steps in 2-3 sentences. Do you think we need more iterations in this particular case? Explain why in one sentence.

</div>

## Your answer here:

---

# 5. Classifying a New Object

Now that we have assigned cluster labels to each datapoint, let's investigate how we should classify a **new** object (which we can see is a Shih-Tzu):

<img src="images/maddie.png"/>

<div class="alert alert-success">

## Exercise 5

Complete the function `assign_new_object` to determine the appropriate cluster label for a new object.

</div>

In [ ]:
def assign_new_object(new_object, updated_centers):
    """
    Returns the cluster label (number) for new_object using k-means 
    clustering.
    
    Parameters
    ----------
    new_object : numpy array of shape (2,)
        The (x,y) coordinates of a new object to be classified
        
    updated_centers : numpy array of shape (num_clusters, 2)
        An array containing the updated (x,y) coordinates for 
        each cluster center
        
    Returns
    -------
    label : int
       The cluster label assignment for new_object. This is a
       number between 0 and (num_clusters - 1).
    """
    # YOUR CODE HERE

In [ ]:
# add your own test cases here!


In [ ]:
"""Check assign_new_object computes the correct values"""
from numpy.testing import assert_equal

# validate using different values
centers1 = np.array([[ 3.17014624,  2.42738134], [ 2.90932354,  4.26426491]])
assert_equal(assign_new_object(np.array([0, 1]), centers1), 0)
assert_equal(assign_new_object(np.array([1, 0]), centers1), 0)
assert_equal(assign_new_object(np.array([3, 2]), centers1), 0)
assert_equal(assign_new_object(np.array([2, 4]), centers1), 1)

centers2 = np.array([[ 3.170146,  2.427381], [ 3.109456,  1.902395], [ 2.964183,  1.827484]])
assert_equal(assign_new_object(np.array([0, 1]), centers2), 2)
assert_equal(assign_new_object(np.array([1, 0]), centers2), 2)
assert_equal(assign_new_object(np.array([3, 2]), centers2), 1)
assert_equal(assign_new_object(np.array([2, 4]), centers2), 0)

# check that it uses distance
old_distance = distance
del distance
try:
    assign_new_object(np.array([3, 2]), centers1)
except NameError:
    pass
else:
    raise AssertionError("assign_new_object does not call distance")
finally:
    distance = old_distance
    del old_distance

print("Success!")

Let's go ahead and rerun k-means to make sure we have the correct variables set:

In [ ]:
# load the data
data = np.load('data/X.npz')
X = data['X']
centers = data['centers'] 

# run k-means
cluster_assignments, updated_centers, fig = k_means(3, X, centers, update_assignments, update_parameters, n_iter=4)

Once you've implemented `assign_new_object`, give it a spin on the image of the Shih-Tzu:

In [ ]:
new_object = np.array([3.3, 3.5])  # image coordinates
label = assign_new_object(new_object, updated_centers)
print('The new object was assigned to cluster: ' + str(label))

Finally, we can visualize this result against the true assignments using the provided helper function `plot_final`:

In [ ]:
def plot_final(X, cluster_assignments, updated_centers, new_object,
               assign_new_object):
    """
    Categorizes a new object and plots it against the true cluster
    labels.
    
    Parameters
    ----------    
    X : numpy array of shape (m, 2)
        An array of m data points in R^2
    
    cluster_assignments : numpy array of shape (m,) 
        The final label assignments for each of the data points in X
    
    updated_centers : numpy array of shape (num_clusters, 2)
        The coordinates for the centers of each cluster after 
        running k_means
        
    new_object : numpy array of shape (2,)
        The (x,y) coordinates of a new object to be classified
    
    assign_new_object : function
        The function you completed in Exercise 5   
        
    Returns
    -------
    fig : matplotlib figure
       The plotted figure.
    """
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    
    # plot data with colors corresponding to cluster assignments
    ax[0].plot(X[(cluster_assignments==0), 0], X[(cluster_assignments==0), 1], 'ro', markersize=8, label='0')
    ax[0].plot(X[(cluster_assignments==1), 0], X[(cluster_assignments==1), 1], 'bo', markersize=8, label='1')
    ax[0].plot(X[(cluster_assignments==2), 0], X[(cluster_assignments==2), 1], 'go', markersize=8, label='2')

    # Generate a label for the new object
    label = assign_new_object(new_object, updated_centers)

    # Plot the new object as a big star on the plot
    colors = ['r', 'b', 'g']
    ax[0].plot(new_object[0], new_object[1], colors[label] + '*', markersize=20,
               markeredgecolor='black', markeredgewidth=1)

    ax[0].set_xlim([2, 4.5])
    ax[0].set_ylim([1, 4.5])
    ax[0].set_aspect('equal')
    ax[0].set_xticks([])
    ax[0].set_yticks([])
    ax[0].set_title('Final Cluster Assignments')
    ax[0].legend()
    
    # Plot the true cluster assignments for comparison
    ax[1].plot(X[:10, 0], X[:10, 1], 'ro', markersize=8, label='cats')
    ax[1].plot(X[10:20, 0], X[10:20, 1], 'bo', markersize=8, label='dogs')
    ax[1].plot(X[20:, 0], X[20:, 1], 'go', markersize=8, label='mops')
    ax[1].set_title('True Cluster Assignments')
    ax[1].set_xlim([2, 4.5])
    ax[1].set_ylim([1, 4.5])
    ax[1].set_aspect('equal')
    ax[1].set_xticks([])
    ax[1].set_yticks([])
    ax[1].legend()
    plt.show()
    
    return fig

In [ ]:
fig = plot_final(X, cluster_assignments, updated_centers, new_object, assign_new_object)

<div class="alert alert-success">

## Exercise 6

When interpreting these plots, don't worry if the coloring differs between the two solutions; what matters is whether k-means identifies the same cluster boundaries as are shown in the true clusters. 

**A.** Explain in one sentence why the colors might differ between the learned and true clusters. *Hint: what do the cluster labels mean in the algorithm?*

**B.** Did the algorithm correctly identify the Shih-Tzu?  
    
**C.** Do you notice any differences between the true clusters and those identified via k-means? Write a few sentences commenting on any differences you found and why these differences might exist.

</div>

## Your answer here:

---

# 6. Random Starting Points

So far, we've used fixed starting points for the cluster centers. In practice, k-means uses **random initialization**. The function `init_centers` below returns random starting points for each cluster.

In [ ]:
def init_centers(k=3):
    """
    Randomly initialize cluster centers.
    
    Parameters
    ----------
    k : int
        Number of clusters
        
    Returns
    -------
    centers : numpy array of shape (k, 2)
        Random initial cluster centers
    """
    return np.vstack([np.random.uniform(low=2., high=4.5, size=k),
                      np.random.uniform(low=1, high=4.5, size=k)]).T

<div class="alert alert-success">

## Exercise 7

Re-run k-means as you did before, but with **random starting points** (using `init_centers()`) instead of the provided starting points, and with **6 iterations** rather than 4. 

Try it out multiple times until your final cluster result is **different** from the one we obtained in Exercise 4. Explain in 2-3 sentences what happened to the algorithm in this case.

</div>

In [ ]:
# YOUR CODE HERE

## Your answer here:

---

# 7. Choosing the Number of Clusters

In this problem, we knew there were three categories (cats, dogs, mops). But in general, you might not know the number of clusters ahead of time. How can we decide what value of $k$ to use?

<div class="alert alert-success">

## Exercise 8

Run k-means with $k=2$ clusters, using **4 iterations** and the first and third of the provided center values as starting points (indices 0 and 2). What pattern do you see in terms of grouping?

</div>

In [ ]:
# load the data
data = np.load('data/X.npz')
X = data['X']
centers = data['centers'][(0, 2), :]  # Use first and third center

# YOUR CODE HERE

## Your answer here:

## Testing on New Data

So far we've only looked at the training data. A better way to evaluate whether k-means has found a meaningful clustering is to test how well it generalizes to **new, held-out data** from the same distributions. We have provided a test dataset (`data/X_test.npz`) containing 150 new points (50 cats, 50 dogs, 50 mops) generated from the same distributions as the original data.

<div class="alert alert-success">

## Exercise 9

We'll test whether k-means can recover the true category structure by measuring classification accuracy on held-out test data.

**A.** Load the test data from `data/X_test.npz`. This file contains:
- `X_test`: 150 test points (50 cats, 50 dogs, 50 mops)
- `y_test`: True labels for each test point (0=cat, 1=dog, 2=mop)

**B.** Visualize the training data and test data side by side.

**C.** For each value of k from 2 to 6:
- Train k-means on the **original training data** (30 points)
- For each cluster, determine which true category it best represents
- Classify the **test points** using the learned cluster centers
- Compute classification accuracy

**D.** Which value of k gives the best accuracy on the test data? Why?

</div>

In [ ]:
# Step A: Load the test data
test_data = np.load('data/X_test.npz')
X_test = test_data['X_test']
y_test = test_data['y_test']

print(f"Test data shape: {X_test.shape}")
print(f"Test labels shape: {y_test.shape}")
print(f"Labels distribution: {np.bincount(y_test)}")

In [ ]:
# YOUR CODE HERE

## Your answer here: